# Model: XGBoost

Owner: **Faiza**

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from xgboost import XGBRegressor

MODEL_NAME = "xgboost"
INCLUDE_RADIATION_DAY_BEFORE = False

In [2]:
# using the shared train/validation/test sets

DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "nsw_train.csv", parse_dates=["DATETIME"])
validation = pd.read_csv(DATA_DIR / "nsw_validation.csv", parse_dates=["DATETIME"])
test = pd.read_csv(DATA_DIR / "nsw_test.csv", parse_dates=["DATETIME"])

print(train.shape, validation.shape, test.shape)

(140208, 55) (17520, 55) (17520, 55)


In [3]:
# TEMPERATURE/radiation are same-day actuals so not usable, only the day_before lags are

TARGET = "TOTALDEMAND"
DROP_COLS = ["DATETIME", TARGET, "TEMPERATURE", "radiation", "forecast_closest", "forecast_12hr_prior", "forecast_dayprior"]
if not INCLUDE_RADIATION_DAY_BEFORE:
    DROP_COLS.append("radiation_day_before")
FEATURES = [c for c in train.columns if c not in DROP_COLS]

len(FEATURES)

47

In [4]:
# hyperparameter combinations for XGBoost
param_grid = [
    {"max_depth": d, "learning_rate": lr, "reg_lambda": reg}
    for d in [3, 6]
    for lr in [0.05, 0.1]
    for reg in [1, 5]
]

best_rmse = None
best_model = None
best_params = None

for params in param_grid:
    xgb = XGBRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
        **params
    )

    xgb.fit(train[FEATURES], train[TARGET])

    val_pred = xgb.predict(validation[FEATURES])
    rmse = np.sqrt(
        mean_squared_error(validation[TARGET], val_pred)
    )

    print(params, f"val rmse: {rmse:.2f}")

    if best_rmse is None or rmse < best_rmse:
        best_rmse = rmse
        best_model = xgb
        best_params = params

print()
print(f"best params: {best_params}, val rmse: {best_rmse:.2f}")

model = best_model

{'max_depth': 3, 'learning_rate': 0.05, 'reg_lambda': 1} val rmse: 458.38
{'max_depth': 3, 'learning_rate': 0.05, 'reg_lambda': 5} val rmse: 458.41
{'max_depth': 3, 'learning_rate': 0.1, 'reg_lambda': 1} val rmse: 449.35
{'max_depth': 3, 'learning_rate': 0.1, 'reg_lambda': 5} val rmse: 448.88
{'max_depth': 6, 'learning_rate': 0.05, 'reg_lambda': 1} val rmse: 434.54
{'max_depth': 6, 'learning_rate': 0.05, 'reg_lambda': 5} val rmse: 434.89
{'max_depth': 6, 'learning_rate': 0.1, 'reg_lambda': 1} val rmse: 433.40
{'max_depth': 6, 'learning_rate': 0.1, 'reg_lambda': 5} val rmse: 431.90

best params: {'max_depth': 6, 'learning_rate': 0.1, 'reg_lambda': 5}, val rmse: 431.90


In [5]:
# refine hyperparameters around the best settings from the first search
refined_grid = [
    {"max_depth": d, "learning_rate": lr, "reg_lambda": reg}
    for d in [5, 6, 7]
    for lr in [0.08, 0.1, 0.12]
    for reg in [0.5, 1, 2]
]

refined_best_rmse = None
refined_best_model = None
refined_best_params = None

for params in refined_grid:
    xgb = XGBRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
        **params
    )

    xgb.fit(train[FEATURES], train[TARGET])

    val_pred = xgb.predict(validation[FEATURES])
    rmse = np.sqrt(
        mean_squared_error(validation[TARGET], val_pred)
    )

    print(params, f"val rmse: {rmse:.2f}")

    if refined_best_rmse is None or rmse < refined_best_rmse:
        refined_best_rmse = rmse
        refined_best_model = xgb
        refined_best_params = params

print()
print(
    f"refined best params: {refined_best_params}, "
    f"val rmse: {refined_best_rmse:.2f}"
)

model = refined_best_model

{'max_depth': 5, 'learning_rate': 0.08, 'reg_lambda': 0.5} val rmse: 436.63
{'max_depth': 5, 'learning_rate': 0.08, 'reg_lambda': 1} val rmse: 437.42
{'max_depth': 5, 'learning_rate': 0.08, 'reg_lambda': 2} val rmse: 436.92
{'max_depth': 5, 'learning_rate': 0.1, 'reg_lambda': 0.5} val rmse: 437.04
{'max_depth': 5, 'learning_rate': 0.1, 'reg_lambda': 1} val rmse: 433.51
{'max_depth': 5, 'learning_rate': 0.1, 'reg_lambda': 2} val rmse: 436.01
{'max_depth': 5, 'learning_rate': 0.12, 'reg_lambda': 0.5} val rmse: 436.16
{'max_depth': 5, 'learning_rate': 0.12, 'reg_lambda': 1} val rmse: 435.71
{'max_depth': 5, 'learning_rate': 0.12, 'reg_lambda': 2} val rmse: 435.01
{'max_depth': 6, 'learning_rate': 0.08, 'reg_lambda': 0.5} val rmse: 433.74
{'max_depth': 6, 'learning_rate': 0.08, 'reg_lambda': 1} val rmse: 433.02
{'max_depth': 6, 'learning_rate': 0.08, 'reg_lambda': 2} val rmse: 432.33
{'max_depth': 6, 'learning_rate': 0.1, 'reg_lambda': 0.5} val rmse: 433.94
{'max_depth': 6, 'learning_rate'

In [6]:
# predict on test now that the model + params are settled using validation above

predictions = model.predict(test[FEATURES])

In [7]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

{'model': 'xgboost', 'rmse': 467.73617716867847, 'mae': 306.6714811599244, 'mape_pct': 3.7275558435638594, 'r2': 0.8599470340577641}


In [8]:
# Save test predictions and performance metrics

output_file = f"{MODEL_NAME}_predictions.csv" if INCLUDE_RADIATION_DAY_BEFORE else f"{MODEL_NAME}_predictions_without_radiation.csv"
pd.Series(predictions, index=test["DATETIME"], name=MODEL_NAME).to_csv(RESULTS_DIR / output_file)

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run

,model,rmse,mae,mape_pct,r2,comments
0,lightgbm,463.995139,307.403004,3.744587,0.862178,NaN
1,baseline_linear_regression,535.989120,380.215288,4.743792,0.816091,\nLinear Regression on the 48 shared features ...
2,prophet,469.383322,345.777006,4.289860,0.843310,NaN
3,random_forest_no_radiation_day_before,502.050043,326.322267,3.962717,0.838644,NaN
4,random_forest,493.696778,322.451412,3.927903,0.843969,NaN
5,aemo_forecast_closest,64.292411,47.711611,0.595688,0.997354,NaN
6,aemo_forecast_12hr_prior,212.913694,155.945260,1.921867,0.970980,NaN
7,aemo_forecast_dayprior,222.460754,161.830989,1.988398,0.968319,NaN
8,lightgbm_no_radiation_day_before,472.457778,311.011912,3.780578,0.857105,NaN
9,xgboost,467.736177,306.671481,3.727556,0.859947,NaN


**XGBoost Model Summary:**


XGBoost was evaluated using the shared data splits in two runs: with 48 features including `radiation_day_before`, and with 47 features excluding it. On the unseen test set, the model with radiation achieved an RMSE of 470.54, MAE of 311.18, MAPE of 3.79% and R² of 0.858. Without radiation, it achieved an RMSE of 467.74, MAE of 306.67, MAPE of 3.73% and R² of 0.860. Performance improved slightly without `radiation_day_before`, indicating that this feature did not improve XGBoost performance during the test period.
